# E4 — Frozen complement-trained matched evaluation

Evaluate six independently trained validation-selected checkpoints on the original five E2-A common populations and exhaustive verification pairs. No training, checkpoint selection or threshold tuning occurs here. Native-annulus footprint removal, exact normalized-zero fill, geometry and normalization match E4 training.

Definitions are self-contained with all imports at the top. Default execution displays recorded results only. Optional numerical replay reads supplied original saved embeddings; fresh inference is a separate explicit option. No other notebook needs to be executed. See [overview](README.md), [report](REPORT.md) and [validation](VALIDATION.md).

## Imports

All package imports are collected in this first code cell.

In [ ]:
from __future__ import annotations

# Standard library
import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
from collections import Counter, defaultdict
from collections.abc import Mapping, Sequence
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from functools import lru_cache
from importlib import metadata
from numbers import Integral
from pathlib import Path, PureWindowsPath

# Scientific libraries, models and notebook display
import torch
import timm
import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib
matplotlib.use("Agg")  # Same backend as the saved scientific figures.
import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.optimize import least_squares
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import auc, roc_curve
from timm.models.swin_transformer_v2 import WindowAttention
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

# Optional: only explicit native-mask generation needs ONNX Runtime.
_ONNXRUNTIME_IMPORT_ERROR = None
try:
    import onnxruntime as ort
except (ImportError, OSError) as error:
    ort = None
    _ONNXRUNTIME_IMPORT_ERROR = error


## Editable paths and execution switches

In [ ]:
REPOSITORY_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents]
    if (p/'Assets/subject_splits.json').is_file()),Path.cwd())
DATA_ROOT=(REPOSITORY_ROOT/'data').resolve()  # EDIT: licensed cache/checkpoint bundle
EVALUATION_MANIFEST=DATA_ROOT/'evaluation_inventory.parquet'
CHECKPOINT_ROOT=DATA_ROOT/'checkpoints'
E4_CHECKPOINT_INDEX=REPOSITORY_ROOT/'Assets/e4_checkpoints.json'
CHECKPOINT_INDEX=REPOSITORY_ROOT/'Assets/checkpoints.json'  # Original E1 index stays separate
SUBJECT_SPLITS=REPOSITORY_ROOT/'Assets/subject_splits.json'
MATCHED_POPULATION=REPOSITORY_ROOT/'Assets/Matched Population'
PATH_REMAP={}
OUTPUT_ROOT=Path('reproduced_outputs/E4-evaluation').resolve()
E4_EMBEDDING_ROOT=DATA_ROOT/'e4_embeddings'  # <dataset>/<job_name>/embeddings.npz
RECORDED_RESULTS=REPOSITORY_ROOT/'E4 Experiments/Results'
INFERENCE_SOURCE='saved_embeddings'  # or explicitly 'recompute'; never an automatic fallback
RUN_ANALYSIS=False
SHOW_RECORDED_RESULTS=True
ANALYSIS_CONFIG={'normalization':{'mean':0.48287347719744544,'std':0.2741831329802273}}
STAGE_CONFIG={'normalization':ANALYSIS_CONFIG['normalization'],
              'support':{'E1-B1':'b1_support','E1-B2':'b2_support'}}


### Scientific settings and environment

In [ ]:
config_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
config_EXTERNAL_DOMAINS = ('AMF', 'UTIRIS_NIR')
config_CONDITIONS = ('E1-A', 'E1-B1', 'E1-B2', 'E1-C1', 'E1-C2')
config_ARCHITECTURES = ('ResNet-18', 'SwinV2-Tiny')
config_SEEDS = (42, 123, 2025)
config_NORMALIZATION = {'mean': 0.48287347719744544, 'std': 0.2741831329802273}

def config_frozen_config(architecture, condition, seed=42, device='cuda'):
    if architecture not in config_ARCHITECTURES or condition not in config_CONDITIONS or seed not in config_SEEDS:
        raise ValueError('Choose the original backbone, E1 condition, and seed 42/123/2025')
    path = None
    config = deepcopy(config_FROZEN_DEFAULTS[architecture][condition])
    config.update(architecture_label=architecture, condition=condition, seed=int(seed))
    config['runtime']['device'] = str(device)
    config['input_shape'] = [1, 64, 512] if condition.startswith('E1-C') else [1, 320, 320]
    config['normalization'] = dict(config_NORMALIZATION)
    return config

def config_environment_info():
    """Display installed versions; do not install packages or change the environment."""
    packages = {}
    for name in ('torch', 'torchvision', 'timm', 'numpy', 'opencv-python', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'nbformat'):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            raise ImportError(f'Missing {name}. Install requirements.txt into the selected notebook kernel environment.') from None
    return {'python': platform.python_version(), **packages, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
config_FROZEN_DEFAULTS = {'ResNet-18': {'E1-A': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1', 'eligibility_policy': 'pure_v2_common_available'}}, 'E1-C2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2', 'eligibility_policy': 'pure_v2_common_available'}}}, 'SwinV2-Tiny': {'E1-A': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1'}}, 'E1-C2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2'}}}}

### Image, mask and tensor transformations

In [ ]:
def preprocessing_read_gray(path, *, native=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing image: {path}. Configure DATA_ROOT/PATH_REMAP or supply the original private cached representation.')
    flags = cv2.IMREAD_GRAYSCALE | cv2.IMREAD_IGNORE_ORIENTATION if native else cv2.IMREAD_UNCHANGED
    image = cv2.imdecode(np.fromfile(path, np.uint8), flags)
    if image is None or image.dtype != np.uint8 or image.ndim != 2:
        raise ValueError(f'Expected unchanged uint8 grayscale pixels: {path}')
    return image

def preprocessing_read_binary(path, shape=None):
    mask = preprocessing_read_gray(path)
    if shape is not None and mask.shape != tuple(shape):
        raise ValueError(f'Image/mask geometry mismatch: {path}')
    if not np.isin(mask, [0, 255]).all():
        raise ValueError(f'Expected an original 0/255 binary mask; no threshold repair: {path}')
    return mask == 255

def preprocessing_geometry(width, height):
    scale = min(320 / width, 320 / height)
    rw, rh = (round(width * scale), round(height * scale))
    left, top = ((320 - rw) // 2, (320 - rh) // 2)
    return {'native_width': width, 'native_height': height, 'content_width': rw, 'content_height': rh, 'pad_left': left, 'pad_top': top, 'pad_right': 320 - rw - left, 'pad_bottom': 320 - rh - top, 'output_width': 320, 'output_height': 320}

def preprocessing_resize_raw(source, plan):
    dimensions = (plan['content_width'], plan['content_height'])
    content = source.copy() if source.shape == (dimensions[1], dimensions[0]) else cv2.resize(source, dimensions, interpolation=cv2.INTER_AREA)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0)

def preprocessing_transform_support(mask, plan):
    if mask.dtype != np.bool_ or mask.shape != (plan['native_height'], plan['native_width']):
        raise ValueError('Native mask and raw-image geometry must match')
    dimensions = (plan['content_width'], plan['content_height'])
    content = mask.astype(np.uint8)
    if content.shape != (dimensions[1], dimensions[0]):
        content = cv2.resize(content, dimensions, interpolation=cv2.INTER_NEAREST_EXACT)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0).astype(bool)

def preprocessing_native_cartesian(gray, masks=None):
    """Exact original letterbox and masks. Never stretch or crop around the iris."""
    plan = preprocessing_geometry(gray.shape[1], gray.shape[0])
    raw = preprocessing_resize_raw(gray, plan)
    if masks is None:
        return (raw, None, None)
    annulus = masks['iris'] & ~masks['pupil']
    return (raw, preprocessing_transform_support(annulus, plan), preprocessing_transform_support(annulus & ~masks['eyelashes'], plan))

def preprocessing_native_polar_v2(gray, masks):
    """Frozen geometry, including unique supported pupil-component resolution.

    Rebuilding is explicit. Geometry failures propagate, never become blank
    strips or V1 fallbacks. C1/C2 share exactly the same coordinates and samples.
    """
    if 'annulus' not in masks:
        masks = dict(masks, annulus=masks['iris'] & ~masks['pupil'])
    strip, valid, occlusion, footprint, x, y, fitted = components_v2_normalize(gray, masks, geometry_v2_configuration())
    clean_valid = valid & ~footprint
    if not clean_valid.any():
        raise ValueError('no_valid_non_lash_tissue: keep this image unavailable to both C1/C2')
    clean = strip.copy()
    clean[~clean_valid] = 0
    return {'c1': strip, 'c1_validity': valid, 'c2': clean, 'c2_validity': clean_valid, 'x': x, 'y': y, 'geometry': fitted, 'eyelash_footprint': footprint, 'eyelash_nearest': occlusion}

def preprocessing_tensor_from_pixels(pixels, condition, normalization, *, support=None, validity=None, augmentation=None):
    image = torch.from_numpy(np.ascontiguousarray(pixels[None])).float().div_(255.0)
    if condition.startswith('E1-C'):
        if validity is None or validity.shape != pixels.shape or (not validity.any()) or pixels[~validity].any():
            raise ValueError('Polar pixels require matching nonempty validity and zero invalid pixels')
        if augmentation is not None:
            image = augmentation_augment_polar(image, torch.from_numpy(validity[None]), **augmentation)
    else:
        if augmentation is not None:
            image = augmentation_PhotometricAugmentation(**augmentation)(image)
        if condition in ('E1-B1', 'E1-B2'):
            if support is None or support.shape != pixels.shape or support.dtype != np.bool_:
                raise ValueError('Cartesian segmented inputs require the corresponding binary support')
            image = image.masked_fill(~torch.from_numpy(support)[None], 0.0)
    image = (image - normalization['mean']) / normalization['std']
    shape = (1, 64, 512) if condition.startswith('E1-C') else (1, 320, 320)
    if image.shape != shape or image.dtype != torch.float32 or (not torch.isfinite(image).all()):
        raise ValueError(f'Expected finite float32 input {shape}, got {tuple(image.shape)}')
    return image

### Frozen inventory, physical-eye identities and representation availability

In [ ]:
data_PATH_COLUMNS = ('native_path', 'raw_path', 'b1_support_path', 'b2_support_path', 'c1_path', 'c1_validity_path', 'c2_path', 'c2_validity_path', 'iris_mask_path', 'pupil_mask_path', 'eyelashes_mask_path')

def data_load_table(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing frozen metadata/pair table: {path}. Supply the documented CSV/Parquet asset; this workflow never invents an inventory or resamples a split.')
    if path.suffix.lower() == '.parquet':
        frame = pd.read_parquet(path)
    elif path.suffix.lower() == '.csv':
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    else:
        raise ValueError(f'Expected CSV or Parquet, got {path}')
    return frame

def data_explicit_bool(value, name):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value in ('true', 'True', '1', 1):
        return True
    if value in ('false', 'False', '0', 0):
        return False
    raise ValueError(f'{name} must be an explicit boolean, got {value!r}')

def data_present(value):
    return value is not None and (not (isinstance(value, float) and np.isnan(value))) and (str(value) != '')

def data_resolve_path(value, data_root, path_remap=None):
    if not data_present(value):
        return None
    text = str(value).replace('\\', '/')
    for old, new in sorted((path_remap or {}).items(), key=lambda item: -len(str(item[0]))):
        prefix = str(old).replace('\\', '/').rstrip('/')
        if text == prefix or text.startswith(prefix + '/'):
            return (Path(new).expanduser() / text[len(prefix):].lstrip('/')).resolve()
    path = Path(text).expanduser()
    if PureWindowsPath(text).is_absolute() and (not path.is_absolute()):
        raise ValueError(f'Foreign absolute path {text!r}. Set PATH_REMAP explicitly; no old user directory is assumed.')
    return (path if path.is_absolute() else Path(data_root) / path).resolve()

def data_load_inventory(manifest, *, data_root, subject_splits, image_exclusions, path_remap=None, strict_original_counts=True):
    """Validate frozen identity/split semantics, never file hashes or source seals."""
    frame = data_load_table(manifest)
    required = {'image_id', 'dataset', 'subject_id', 'eye_side', 'iris_id', 'source_id', 'split'}
    if (missing := (required - set(frame))):
        raise ValueError(f'Image manifest lacks {sorted(missing)}. See ASSETS.md for the portable schema; source_id must be the original canonical-content identifier.')
    frozen = json.loads(Path(subject_splits).read_text(encoding='utf-8'))
    exclusions = json.loads(Path(image_exclusions).read_text(encoding='utf-8'))
    if frozen['seed'] != 42 or set(frozen['datasets']) != set(config_DOMAINS):
        raise ValueError('Supply the original seed-42 subject allocations for the three primary datasets')
    if strict_original_counts:
        canonical = data_FROZEN_MEMBERSHIP

        def allocations(value):
            return {d: {s: sorted(map(str, ids)) for s, ids in splits.items()} for d, splits in value.items()}
        if allocations(frozen['datasets']) != allocations(canonical['datasets']):
            raise ValueError('Supplied labels change the original seed-42 subject allocations; no replacement split is permitted')
        if {r['image_id'] for r in exclusions['excluded_images']} != set(canonical['excluded_image_ids']) or {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']} != {(r['dataset'], r['subject_id']) for r in canonical['excluded_subjects']}:
            raise ValueError('Supplied image/subject exclusions differ from the original approved exclusions')
    assignment = {}
    for domain, partitions in frozen['datasets'].items():
        for split, subjects in partitions.items():
            for subject in subjects:
                key = (domain, str(subject))
                if key in assignment:
                    raise ValueError(f'Frozen subject occurs in multiple partitions: {key}')
                assignment[key] = split
    excluded_ids = {r['image_id'] for r in exclusions['excluded_images']}
    excluded_subjects = {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']}
    rows, excluded = ([], [])
    seen = set()
    for row in frame.to_dict('records'):
        for key in required:
            if not data_present(row[key]):
                raise ValueError(f'Missing {key}; labels/source identity may not be guessed')
            row[key] = str(row[key])
        if row['image_id'] in seen:
            raise ValueError(f"Duplicate image ID: {row['image_id']}")
        seen.add(row['image_id'])
        if row['eye_side'] not in ('L', 'R') or row['iris_id'] != f"{row['dataset']}:{row['subject_id']}:{row['eye_side']}":
            raise ValueError(f"Identity/eye namespace mismatch: {row['image_id']}")
        key = (row['dataset'], row['subject_id'])
        if row['image_id'] in excluded_ids or key in excluded_subjects:
            excluded.append(row['image_id'])
            continue
        if row['dataset'] in config_DOMAINS:
            if key not in assignment or row['split'] != assignment[key]:
                raise ValueError(f'Manifest changes the frozen dataset+subject partition: {key}')
        elif row['dataset'] in config_EXTERNAL_DOMAINS:
            if row['split'] != 'external':
                raise ValueError('AMF and UTIRIS_NIR must remain external, never training/validation')
        else:
            raise ValueError(f"Dataset outside E1 scope: {row['dataset']}")
        if 'included' in row and data_present(row['included']) and (not data_explicit_bool(row['included'], 'included')):
            raise ValueError(f"Unapproved additional exclusion: {row['image_id']}. Supply the retained frozen inventory, including QC flags.")
        for column in data_PATH_COLUMNS:
            row[column] = data_resolve_path(row.get(column), data_root, path_remap)
        row['quality_flags'] = row.get('quality_flags', [])
        rows.append(row)
    if not rows:
        raise ValueError('No retained images in the supplied frozen inventory')
    sources = defaultdict(list)
    for row in rows:
        sources[row['source_id']].append(row)
    duplicates = {source: group for source, group in sources.items() if len(group) > 1}
    if duplicates:
        raise ValueError('Redundant canonical source images remain in the included inventory; use the existing audited exact-duplicate exclusions, not a new deduplication rule')
    if strict_original_counts:
        expected = data_INVENTORY_COUNTS
        actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS))
        required_counts = {(d, s): n for d, splits in expected['raw'].items() for s, n in splits.items()}
        if actual != required_counts:
            raise ValueError('Portable manifest is not the complete frozen primary inventory. Supply the original author metadata (all retained train/validation/test rows); do not invent a subset or replacement split. Synthetic software tests must explicitly disable this count check.')
        if 'c1_status' in frame and 'c2_status' in frame:
            actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS and r['c1_status'] == r['c2_status'] == 'ok'))
            required_counts = {(d, s): n for d, splits in expected['polar_common'].items() for s, n in splits.items()}
            if actual != required_counts:
                raise ValueError('Pure-V2 common availability differs from the original audit; supply the frozen status/eligibility manifest rather than new exclusions')
    return (sorted(rows, key=lambda r: r['image_id']), {'images': len(rows), 'approved_exclusion_rows_filtered': excluded, 'groups': dict(Counter(((r['dataset'], r['split']) for r in rows))), 'new_subject_assignments': False, 'file_hash_checks': False})

def data_condition_view(records, condition, split):
    """Keep the historical common C1/C2 availability; report every failure."""
    if split not in ('train', 'validation', 'test', 'external'):
        raise ValueError('Unknown split')
    selected, unavailable = ([], [])
    for row in records:
        if row['split'] != split:
            continue
        if condition in ('E1-B1', 'E1-B2'):
            key = 'b1_status' if condition == 'E1-B1' else 'b2_status'
            if key in row and row[key] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row[key]})
                continue
        if condition.startswith('E1-C'):
            if not all((name in row for name in ('c1_status', 'c2_status', 'geometry_version', 'v1_fallback_used', 'v3_used'))):
                raise ValueError('Polar reproduction requires the frozen pure-V2 availability/status columns and C1/C2 assets; supply the portable V2 manifest, not guessed geometry')
            if str(row['geometry_version']) not in ('v2', '2') or data_explicit_bool(row['v1_fallback_used'], 'v1_fallback_used') or data_explicit_bool(row['v3_used'], 'v3_used'):
                raise ValueError('Only pure V2 is allowed; no V1/hybrid/V3 substitutions')
            if row['c1_status'] != 'ok' or row['c2_status'] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row.get('failure_reason', 'V2 common availability')})
                continue
            key = 'c1' if condition == 'E1-C1' else 'c2'
            if row.get(key + '_path') is None or row.get(key + '_validity_path') is None:
                raise FileNotFoundError(f"Missing frozen {key.upper()} texture/validity for {row['image_id']}. Supply the original cache or explicitly rebuild using the frozen V2 method and audited availability.")
        selected.append(dict(row))
    if not selected:
        raise ValueError(f'No eligible {split} images for {condition}')
    return (selected, unavailable)

def data_pk_eligible(records):
    sources = defaultdict(set)
    for row in records:
        if row['split'] != 'train':
            raise ValueError('PK eligibility must use training records only')
        sources[row['iris_id']].add(row['source_id'])
    return [r for r in records if len(sources[r['iris_id']]) >= 2]

class data_IrisDataset(Dataset):
    """Exact cached/native tensors with explicit split access and no resizing of caches."""

    def __init__(self, records, config, *, split, augment=False):
        self.records = list(records)
        self.condition = config['condition']
        self.normalization = dict(config['normalization'])
        self.evaluation = not augment
        if not self.records or any((r['split'] != split for r in self.records)):
            raise ValueError('Dataset access requires one explicit, matching partition')
        if augment and split != 'train':
            raise ValueError('Augmentation is permitted on training captures only')
        aug = config['augmentation']
        self.augmentation = {k: aug[k] for k in ('probability', 'brightness', 'contrast')} if augment and aug['enabled'] else None
        self.dataset_labels = {d: i for i, d in enumerate((*config_DOMAINS, *config_EXTERNAL_DOMAINS))}
        self.subject_labels = {k: i for i, k in enumerate(sorted({(r['dataset'], r['subject_id']) for r in self.records}))}
        self.iris_labels = {k: i for i, k in enumerate(sorted({r['iris_id'] for r in self.records}))}
        self.source_labels = {k: i for i, k in enumerate(sorted({r['source_id'] for r in self.records}))}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        support = validity = None
        native_shape = (int(row['height']), int(row['width'])) if data_present(row.get('height')) and data_present(row.get('width')) else None
        if self.condition.startswith('E1-C'):
            key = 'c1' if self.condition == 'E1-C1' else 'c2'
            pixels = preprocessing_read_gray(row[key + '_path'])
            validity = preprocessing_read_binary(row[key + '_validity_path'], (64, 512))
        else:
            if row.get('raw_path') is not None:
                pixels = preprocessing_read_gray(row['raw_path'])
                if pixels.shape != (320, 320):
                    raise ValueError('Cached raw pixels must already be 320x320; no extra resize is allowed')
            else:
                if row.get('native_path') is None:
                    raise FileNotFoundError('Provide raw_path or native_path in the image manifest')
                native = preprocessing_read_gray(row['native_path'], native=True)
                if native_shape is not None and native.shape != native_shape:
                    raise ValueError('Native image dimensions disagree with the authoritative inventory')
                native_shape = native.shape
                pixels = preprocessing_resize_raw(native, preprocessing_geometry(native.shape[1], native.shape[0]))
            if self.condition in ('E1-B1', 'E1-B2'):
                column = 'b1_support_path' if self.condition == 'E1-B1' else 'b2_support_path'
                if row.get(column) is not None:
                    support = preprocessing_read_binary(row[column], (320, 320))
                else:
                    if any((row.get(k + '_mask_path') is None for k in ('iris', 'pupil', 'eyelashes'))):
                        raise FileNotFoundError('Provide the frozen support cache or the native iris/pupil/eyelash masks; no segmentation model is substituted')
                    masks = {k: preprocessing_read_binary(row[k + '_mask_path']) for k in ('iris', 'pupil', 'eyelashes')}
                    if len({m.shape for m in masks.values()}) != 1:
                        raise ValueError('Native segmentation masks do not align')
                    if native_shape is None:
                        if row.get('native_path') is None:
                            raise ValueError('Native-mask fallback needs original width/height or native_path to verify alignment with the raw frame')
                        native_shape = preprocessing_read_gray(row['native_path'], native=True).shape
                    if masks['iris'].shape != native_shape:
                        raise ValueError('Native mask/image geometry mismatch; do not resize mismatched anatomy into apparent alignment')
                    native_support = masks['iris'] & ~masks['pupil']
                    if self.condition == 'E1-B2':
                        native_support &= ~masks['eyelashes']
                    support = preprocessing_transform_support(native_support, preprocessing_geometry(native_support.shape[1], native_support.shape[0]))
        image = preprocessing_tensor_from_pixels(pixels, self.condition, self.normalization, support=support, validity=validity, augmentation=self.augmentation)
        scalar = lambda x: torch.tensor(x, dtype=torch.long)
        return {'image': image, 'dataset_id': scalar(self.dataset_labels[row['dataset']]), 'subject_label': scalar(self.subject_labels[row['dataset'], row['subject_id']]), 'eye_label': scalar(0 if row['eye_side'] == 'L' else 1), 'iris_label': scalar(self.iris_labels[row['iris_id']]), 'source_label': scalar(self.source_labels[row['source_id']]), **{k: row[k] for k in ('image_id', 'iris_id', 'dataset', 'subject_id', 'eye_side', 'split')}}
data_FROZEN_MEMBERSHIP = {'seed': 42, 'datasets': {'CASIA-Interval': {'test': ['009', '010', '018', '023', '025', '028', '039', '044', '057', '070', '074', '096', '099', '101', '114', '125', '130', '131', '141', '149', '152', '153', '165', '175', '176', '182', '188', '191', '204', '205', '212', '218', '219', '231', '233', '234', '249'], 'train': ['001', '003', '004', '006', '007', '008', '011', '012', '013', '014', '015', '016', '019', '020', '021', '024', '026', '030', '032', '033', '034', '036', '037', '038', '040', '041', '042', '043', '045', '046', '047', '048', '051', '052', '053', '054', '055', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '071', '072', '073', '075', '076', '077', '078', '080', '081', '083', '084', '085', '086', '087', '089', '090', '092', '093', '095', '097', '102', '105', '106', '107', '108', '110', '111', '112', '113', '116', '117', '118', '119', '120', '121', '123', '124', '126', '127', '128', '129', '132', '133', '134', '135', '136', '137', '138', '139', '140', '142', '143', '144', '146', '147', '148', '150', '151', '154', '155', '156', '157', '158', '159', '160', '161', '163', '164', '166', '167', '168', '169', '170', '171', '172', '173', '178', '179', '181', '184', '185', '186', '187', '189', '190', '192', '193', '194', '195', '196', '197', '198', '200', '201', '202', '203', '206', '207', '209', '210', '211', '213', '214', '215', '216', '222', '223', '224', '225', '226', '227', '228', '229', '230', '232', '236', '237', '240', '241', '242', '243', '244', '245', '246', '248'], 'validation': ['002', '005', '017', '022', '027', '029', '031', '035', '049', '050', '056', '079', '082', '088', '091', '094', '098', '100', '103', '104', '109', '115', '122', '145', '162', '174', '177', '180', '183', '199', '208', '217', '220', '221', '235', '238', '239', '247']}, 'CUST-Iris': {'test': ['11', '17', '23', '28', '109', '114', '116', '120', '141', '142', '143', '147', '158', '173', '174', '176', '200', '216', '224', '226', '228', '244', '247', '248', '251', '252', '259', '264', '272', '279', '286', '295', '296', '302', '309', '313', '314', '320', '334', '355', '356', '359', '364', '365', '372', '376', '379', '382', '387', '393', '395', '413', '417', '423'], 'train': ['1', '2', '10', '12', '13', '14', '15', '16', '19', '21', '22', '24', '25', '27', '29', '30', '31', '32', '33', '34', '35', '37', '38', '39', '40', '41', '100', '102', '103', '104', '106', '108', '110', '111', '112', '113', '115', '118', '119', '121', '123', '124', '125', '126', '128', '129', '131', '132', '133', '134', '136', '137', '138', '139', '144', '145', '146', '148', '149', '150', '151', '154', '155', '156', '157', '159', '160', '161', '162', '163', '164', '165', '166', '167', '169', '170', '171', '172', '175', '177', '178', '179', '180', '181', '182', '183', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '201', '202', '203', '205', '206', '207', '209', '210', '211', '214', '215', '217', '218', '219', '220', '221', '222', '223', '225', '227', '229', '230', '232', '233', '234', '235', '236', '238', '239', '241', '242', '243', '245', '246', '250', '254', '255', '256', '257', '258', '260', '262', '263', '265', '266', '267', '269', '271', '273', '275', '276', '277', '278', '280', '281', '282', '283', '284', '285', '287', '288', '289', '290', '291', '292', '294', '297', '298', '299', '300', '301', '304', '307', '308', '310', '311', '312', '316', '317', '318', '319', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '336', '337', '339', '340', '341', '342', '343', '344', '345', '347', '348', '349', '352', '354', '360', '361', '362', '363', '366', '367', '368', '369', '370', '371', '373', '374', '375', '377', '378', '380', '381', '383', '386', '388', '389', '391', '392', '394', '397', '399', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412', '415', '416', '418', '419', '420', '421', '422', '424'], 'validation': ['3', '18', '20', '26', '36', '101', '105', '107', '117', '122', '127', '130', '135', '140', '152', '153', '168', '184', '185', '199', '204', '208', '212', '213', '231', '237', '240', '249', '253', '261', '268', '270', '274', '293', '303', '305', '306', '315', '321', '335', '338', '346', '350', '351', '353', '357', '358', '384', '385', '390', '396', '398', '400', '414']}, 'IITD': {'test': ['004', '013', '021', '026', '033', '034', '042', '059', '061', '064', '070', '072', '074', '081', '088', '092', '115', '117', '118', '128', '135', '153', '158', '166', '169', '174', '178', '189', '192', '198', '206', '214'], 'train': ['002', '003', '005', '007', '008', '009', '010', '011', '012', '014', '015', '016', '017', '018', '019', '020', '023', '024', '025', '027', '029', '030', '032', '035', '036', '037', '038', '039', '040', '041', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '055', '057', '060', '062', '063', '065', '066', '067', '068', '069', '076', '077', '078', '079', '080', '082', '083', '084', '085', '086', '091', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '105', '106', '107', '108', '109', '112', '113', '114', '116', '119', '120', '121', '122', '123', '126', '127', '129', '130', '131', '132', '133', '137', '138', '139', '140', '142', '143', '144', '145', '146', '147', '148', '149', '152', '154', '155', '156', '157', '159', '160', '162', '163', '164', '165', '167', '168', '170', '171', '172', '175', '176', '179', '181', '182', '183', '184', '185', '186', '187', '188', '190', '191', '195', '196', '197', '199', '200', '201', '202', '203', '204', '207', '208', '209', '210', '211', '212', '213', '215', '217'], 'validation': ['006', '022', '028', '031', '054', '056', '058', '071', '073', '075', '087', '089', '090', '093', '104', '110', '111', '124', '125', '134', '136', '141', '150', '151', '161', '173', '177', '180', '193', '194', '205', '216', '218']}}, 'excluded_image_ids': ['IITD:001_01_L', 'IITD:001_02_L', 'IITD:001_03_L', 'IITD:001_04_L', 'IITD:001_05_L', 'IITD:001_06_L', 'IITD:001_06_R', 'IITD:001_07_L', 'IITD:001_07_R', 'IITD:001_08_L', 'IITD:001_08_R', 'IITD:001_09_L', 'IITD:001_09_R', 'IITD:001_10_L', 'IITD:001_10_R', 'IITD:018_08_R', 'IITD:038_02_L', 'IITD:111_02_L', 'IITD:128_10_R', 'IITD:140_09_R', 'IITD:150_02_L', 'IITD:185_10_R', 'IITD:207_06_L', 'IITD:212_06_L', 'IITD:213_07_L'], 'excluded_subjects': [{'dataset': 'IITD', 'subject_id': '001'}]}
data_INVENTORY_COUNTS = {'raw': {'CASIA-Interval': {'train': 1847, 'validation': 396, 'test': 396}, 'CUST-Iris': {'train': 2016, 'validation': 432, 'test': 432}, 'IITD': {'train': 1509, 'validation': 327, 'test': 319}}, 'polar_common': {'CASIA-Interval': {'train': 1842, 'validation': 394, 'test': 395}, 'CUST-Iris': {'train': 2002, 'validation': 426, 'test': 427}, 'IITD': {'train': 1476, 'test': 308, 'validation': 318}}}

### ResNet-18 shared embedding encoder

In [ ]:
class resnet_IrisEmbeddingEncoder(nn.Module):

    def __init__(self, pretrained: bool=True, pretrained_file=None) -> None:
        super().__init__()
        self.backbone = timm.create_model('resnet18', pretrained=pretrained, num_classes=0, in_chans=1, global_pool='avg', **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        if self.backbone.num_features != 512:
            raise ValueError('Expected the 512-dimensional ResNet18 backbone')
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(512, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 320, 320):
            raise ValueError(f'Expected [B,1,320,320], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

### SwinV2-Tiny shared embedding encoder

In [ ]:
swin_MODEL_NAME = 'swinv2_tiny_window8_256'
swin_PRETRAINED_TAG = 'ms_in1k'
swin_ALLOWED_SHAPES = ((320, 320), (64, 512))

class swin_Float32WindowAttention(WindowAttention):
    """User-approved FP32 attention; preserve timm operations and parameter keys.

    Padded zero tokens produce nonfinite FP16 cosine-attention gradients in the
    installed timm implementation. Disable autocast for this module only. The
    patch embedding, MLPs, other backbone operations and head still use AMP.
    """

    def forward(self, x, mask=None):
        with torch.autocast(device_type=x.device.type, enabled=False):
            return super().forward(x.float(), None if mask is None else mask.float())

def swin_use_float32_attention(model):
    """Change execution only, without initializing tensors or consuming RNG."""
    modules = [module for module in model.modules() if type(module) is WindowAttention]
    if len(modules) != 12:
        raise ValueError(f'Expected 12 native SwinV2 attention modules, got {len(modules)}')
    for module in modules:
        module.__class__ = swin_Float32WindowAttention
    return len(modules)

class swin_SwinBackbone(nn.Module):
    """Expose the first patch convolution to the unchanged E1 gradient probe.

    The property does not register another module or duplicate any parameters.
    Windowing, pooling and pretrained loading stay in timm. Attention uses the
    approved float32 execution context around timm's unchanged calculation.
    """

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.model = timm.create_model(swin_MODEL_NAME, pretrained=pretrained, num_classes=0, in_chans=1, img_size=tuple(img_size), global_pool='avg', strict_img_size=False, **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        self.float32_attention_modules = swin_use_float32_attention(self.model)
        self.num_features = int(self.model.num_features)
        if self.num_features != 768:
            raise ValueError(f'Unexpected SwinV2-Tiny features: {self.num_features}')
        if self.model.pretrained_cfg['tag'] != swin_PRETRAINED_TAG:
            raise ValueError('Unexpected pretrained tag')

    @property
    def conv1(self):
        return self.model.patch_embed.proj

    def forward(self, images):
        features = self.model(images)
        if features.ndim != 2 or features.shape != (images.shape[0], self.num_features):
            raise ValueError(f'Expected pooled [B,{self.num_features}], got {tuple(features.shape)}')
        return features

class swin_SwinEmbeddingEncoder(nn.Module):

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.img_size = tuple(img_size)
        if self.img_size not in swin_ALLOWED_SHAPES:
            raise ValueError(f'Unapproved E1 input shape: {self.img_size}')
        self.backbone = swin_SwinBackbone(self.img_size, pretrained=pretrained, pretrained_file=pretrained_file)
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(self.backbone.num_features, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, *self.img_size):
            raise ValueError(f'Expected [B,1,{self.img_size[0]},{self.img_size[1]}], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def swin_describe_windows(encoder):
    return [dict(stage=i, input_resolution=list(stage.input_resolution), output_resolution=list(stage.output_resolution), windows=[list(block.window_size) for block in stage.blocks], shifts=[list(block.shift_size) for block in stage.blocks], dynamic_masks=[bool(block.dynamic_mask) for block in stage.blocks]) for i, stage in enumerate(encoder.backbone.model.layers)]

### Model construction and strict input guards

In [ ]:
class models_PolarEncoder(resnet_IrisEmbeddingEncoder):
    """Original ResNet polar adapter: only the spatial guard differs."""

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 64, 512):
            raise ValueError('Expected [B,1,64,512] polar input')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def models_build_model(config, *, pretrained=True, pretrained_asset=None, allow_download=False):
    """Initialize independently, using the original timm RGB-to-gray adaptation.

    `pretrained=False` is reserved for clearly labelled software tests or loading
    a complete trained checkpoint, never a fallback after failed asset loading.
    """
    architecture, condition = (config['architecture_label'], config['condition'])
    if pretrained_asset is not None:
        pretrained_asset = Path(pretrained_asset).expanduser().resolve()
        if not pretrained_asset.is_file():
            raise FileNotFoundError(f'Missing pretrained asset: {pretrained_asset}. Download the documented original timm model.safetensors and set PRETRAINED_ASSET.')
    if pretrained and pretrained_asset is None and (not allow_download):
        raise FileNotFoundError('Provide PRETRAINED_ASSET or explicitly set ALLOW_PRETRAINED_DOWNLOAD=True. No random-weight fallback is used.')
    try:
        if architecture == 'ResNet-18':
            cls = models_PolarEncoder if condition.startswith('E1-C') else resnet_IrisEmbeddingEncoder
            model = cls(pretrained=pretrained, pretrained_file=pretrained_asset)
        elif architecture == 'SwinV2-Tiny':
            model = swin_SwinEmbeddingEncoder(config['input_shape'][1:], pretrained=pretrained, pretrained_file=pretrained_asset)
        else:
            raise ValueError(f'Unsupported architecture: {architecture}')
    except (OSError, RuntimeError) as error:
        raise RuntimeError(f'Could not initialize the original {architecture} pretrained source. Check the documented tagged asset and installed timm version. No alternative weights were substituted. {error}') from error
    if not all((p.requires_grad for p in model.parameters())):
        raise RuntimeError('All backbone and projection parameters must remain trainable')
    return model

def models_model_summary(model, config):
    return {'architecture': config['architecture_label'], 'condition': config['condition'], 'input_shape': ['B', *config['input_shape']], 'output_shape': ['B', 256], 'pooled_features': model.backbone.num_features, 'trainable_parameters': sum((p.numel() for p in model.parameters() if p.requires_grad)), 'single_shared_encoder': True, 'l2_normalized_output': True, 'swin_float32_window_attention': config['architecture_label'] == 'SwinV2-Tiny'}

### Complete ROC, EER and low-FAR metric definitions

In [ ]:
metrics_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
metrics_EER_METHOD = 'Linear interpolation on the complete ROC: locate adjacent points where d=FNR-FPR crosses zero; alpha=d0/(d0-d1), EER=FPR0+alpha*(FPR1-FPR0). Tied scores move together. No fixed operating threshold is selected.'
metrics_TAR_METHOD = 'ROC-interpolated TAR on the complete ROC: at an exact repeated FPR select its maximum TPR; between knots interpolate from the last lower-FPR point to the first higher-FPR point, preserving future vertical jumps. These values are not empirical results at a fixed threshold.'

def metrics_scores(values, name: str) -> np.ndarray:
    result = np.asarray(values, dtype=np.float64)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError(f'{name} must be a nonempty finite one-dimensional score array')
    return result

def metrics_interpolated_eer(fpr: np.ndarray, tpr: np.ndarray) -> float:
    difference = 1.0 - tpr - fpr
    right = int(np.flatnonzero(difference <= 0)[0])
    if difference[right] == 0 or right == 0:
        return float(fpr[right])
    left = right - 1
    alpha = difference[left] / (difference[left] - difference[right])
    return float(fpr[left] + alpha * (fpr[right] - fpr[left]))

def metrics_interpolated_tar(fpr: np.ndarray, tpr: np.ndarray, far: float) -> float:
    right = int(np.searchsorted(fpr, far, side='right'))
    left = right - 1
    if right == len(fpr):
        return float(tpr[-1])
    if left < 0:
        return float(tpr[0])
    if fpr[left] == far:
        return float(tpr[left])
    alpha = (far - fpr[left]) / (fpr[right] - fpr[left])
    return float(tpr[left] + alpha * (tpr[right] - tpr[left]))

def metrics_verification_metrics(genuine_scores, impostor_scores) -> dict:
    """Return float64 metrics from scores for both classes, without tuning.

    AUC is the trapezoidal area of the full ROC. EER is obtained by interpolating
    the FNR-FPR zero crossing. FAR resolution is one false acceptance divided by
    the number of impostor comparisons; interpolation does not improve that
    empirical resolution or establish independent-trial confidence intervals.
    """
    genuine = metrics_scores(genuine_scores, 'Genuine scores')
    impostor = metrics_scores(impostor_scores, 'Impostor scores')
    labels = np.concatenate((np.ones(len(genuine), dtype=np.uint8), np.zeros(len(impostor), dtype=np.uint8)))
    scores = np.concatenate((genuine, impostor))
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1, drop_intermediate=False)
    resolution = 1.0 / len(impostor)
    notes = []
    for far in (0.01, 0.001):
        if far < resolution:
            notes.append(f'FAR={far:g} is below one observed false acceptance ({resolution:.12g}); its TAR is an ROC interpolation, not an empirically resolved fixed-threshold operating point.')
    return {'roc_auc': float(auc(fpr, tpr)), 'eer': metrics_interpolated_eer(fpr, tpr), 'tar_at_far_1pct': metrics_interpolated_tar(fpr, tpr, 0.01), 'tar_at_far_0_1pct': metrics_interpolated_tar(fpr, tpr, 0.001), 'num_genuine': int(len(genuine)), 'num_impostor': int(len(impostor)), 'far_resolution': resolution, 'eer_method': metrics_EER_METHOD, 'tar_method': metrics_TAR_METHOD, 'tar_is_roc_interpolated': True, 'low_far_notes': notes}

def metrics_numpy_embedding(value) -> np.ndarray:
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    result = np.asarray(value, dtype=np.float32)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError('Each embedding must be a finite nonempty vector')
    return result

def metrics_pair_chunks(pairs, chunk_size: int):
    if hasattr(pairs, 'to_batches'):
        for batch in pairs.to_batches(max_chunksize=chunk_size):
            yield batch.to_pylist()
    elif isinstance(pairs, Sequence) and (not isinstance(pairs, (str, bytes))):
        for offset in range(0, len(pairs), chunk_size):
            yield pairs[offset:offset + chunk_size]
    else:
        raise TypeError('Pairs must be a sequence of dictionaries or a PyArrow Table')

def metrics_evaluate_validation(embeddings_by_id: Mapping, pairs_by_dataset: Mapping, chunk_size: int=65536) -> dict:
    """Score every frozen pair and macro-average the three validation EERs.

    No images are opened here. Embeddings are CPU float32 vectors, converted
    from tensors if needed and normalized before cosine dot products. Scoring
    is chunked without choosing, discarding or sampling pairs based on scores.
    Pair membership/subject/source eligibility is sealed by the input protocol;
    this function additionally rejects self-comparisons and domain mismatches.
    """
    if not isinstance(chunk_size, Integral) or chunk_size <= 0:
        raise ValueError('chunk_size must be a positive integer')
    if set(pairs_by_dataset) != set(metrics_DOMAINS):
        raise ValueError('Macro-validation requires exactly CUST-Iris, CASIA-Interval and IITD')
    if not embeddings_by_id:
        raise ValueError('Validation embeddings cannot be empty')
    ids = sorted(embeddings_by_id)
    vectors = [metrics_numpy_embedding(embeddings_by_id[key]) for key in ids]
    if len({v.shape for v in vectors}) != 1:
        raise ValueError('Validation embeddings must share one feature dimension')
    matrix = np.stack(vectors)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if not np.isfinite(norms).all() or (norms <= 0).any():
        raise ValueError('Validation embeddings must have finite positive norms')
    matrix = matrix / norms
    index = {key: i for i, key in enumerate(ids)}
    per_dataset = {}
    for domain in metrics_DOMAINS:
        genuine_parts, impostor_parts = ([], [])
        for rows in metrics_pair_chunks(pairs_by_dataset[domain], int(chunk_size)):
            left, right, genuine = ([], [], [])
            for row in rows:
                if row['dataset'] != domain:
                    raise ValueError('Pair dataset differs from its frozen domain group')
                a, b = (row['image_id_a'], row['image_id_b'])
                if a == b:
                    raise ValueError('Self/identical-image comparisons are forbidden')
                if a not in index or b not in index:
                    raise ValueError(f'Frozen validation pair lacks an embedding: {a}, {b}')
                if not isinstance(row['is_genuine'], (bool, np.bool_)):
                    raise ValueError('Pair is_genuine must be an explicit boolean')
                left.append(index[a])
                right.append(index[b])
                genuine.append(row['is_genuine'])
            if not left:
                continue
            similarities = np.einsum('ij,ij->i', matrix[left], matrix[right], dtype=np.float32)
            similarities = np.clip(similarities, -1.0, 1.0)
            mask = np.asarray(genuine, dtype=bool)
            genuine_parts.append(similarities[mask])
            impostor_parts.append(similarities[~mask])
        genuine_scores = np.concatenate(genuine_parts) if genuine_parts else np.empty(0)
        impostor_scores = np.concatenate(impostor_parts) if impostor_parts else np.empty(0)
        per_dataset[domain] = metrics_verification_metrics(genuine_scores, impostor_scores)
    return {'per_dataset': per_dataset, 'macro_eer': float(np.mean([per_dataset[d]['eer'] for d in metrics_DOMAINS], dtype=np.float64)), 'score': 'cosine similarity of L2-normalized float32 embeddings', 'selection_metric': 'unweighted macro-average of three per-dataset validation EERs'}

### Physical-eye verification-pair checks

In [ ]:
pairs_KINDS = ('genuine', 'impostor', 'opposite_eye_diagnostic')

def pairs_validate_pairs(rows, records, *, domain, split, allow_availability_subset=False):
    by_id = {r['image_id']: r for r in records}
    pairs, unavailable, seen = ([], [], set())
    for raw in rows:
        row = dict(raw)
        a, b = (str(row['image_id_a']), str(row['image_id_b']))
        if a not in by_id or b not in by_id:
            if allow_availability_subset:
                unavailable.append((a, b))
                continue
            raise ValueError(f'Frozen pair lacks a member in its declared population: {a}, {b}')
        left, right = (by_id[a], by_id[b])
        if left['split'] != split or right['split'] != split or left['dataset'] != domain or (right['dataset'] != domain):
            raise ValueError('Pair crosses a dataset or frozen partition boundary')
        if a == b or left['source_id'] == right['source_id']:
            raise ValueError('Self/identical-source images may not count as verification pairs')
        kind = row.get('pair_type')
        if not kind:
            kind = 'genuine' if data_explicit_bool(row['is_genuine'], 'is_genuine') else 'impostor'
        if kind not in pairs_KINDS:
            raise ValueError(f'Unexpected pair category: {kind}')
        genuine = left['iris_id'] == right['iris_id']
        impostor = left['subject_id'] != right['subject_id']
        opposite = left['subject_id'] == right['subject_id'] and left['eye_side'] != right['eye_side']
        if not {'genuine': genuine, 'impostor': impostor, 'opposite_eye_diagnostic': opposite}[kind]:
            raise ValueError(f'Pair labels disagree with dataset+subject+eye: {a}, {b}')
        if 'is_genuine' in row and data_explicit_bool(row['is_genuine'], 'is_genuine') != genuine:
            raise ValueError("Pair's is_genuine contradicts physical-eye identity")
        if 'label' in row and int(row['label']) != int(genuine):
            raise ValueError("Pair's label contradicts physical-eye identity")
        if 'dataset' in row and row['dataset'] != domain:
            raise ValueError("Pair's dataset disagrees with its members")
        if 'split' in row and row['split'] != split:
            raise ValueError("Pair's split disagrees with its members")
        if 'primary_evaluation' in row and data_explicit_bool(row['primary_evaluation'], 'primary_evaluation') != (kind != 'opposite_eye_diagnostic'):
            raise ValueError("Pair's primary/diagnostic flag contradicts its category")
        for side, member in (('a', left), ('b', right)):
            expected_fields = {f'iris_id_{side}': member['iris_id'], f'eye_side_{side}': member['eye_side'], f'partition_key_{side}': f"{domain}:{member['subject_id']}"}
            for key, expected in expected_fields.items():
                if key in row and row[key] != expected:
                    raise ValueError(f"Frozen pair's {key} contradicts the authoritative image label")
        key = tuple(sorted((a, b)))
        if key in seen:
            raise ValueError('Repeated/reversed pair in a frozen population')
        seen.add(key)
        pairs.append({**row, 'image_id_a': a, 'image_id_b': b, 'dataset': domain, 'split': split, 'pair_type': kind, 'is_genuine': bool(genuine), 'primary_evaluation': kind != 'opposite_eye_diagnostic'})
    return (pairs, unavailable)

def pairs_load_validation_pairs(files, records, condition, *, original_validation_records=None):
    if set(files) != set(config_DOMAINS):
        raise ValueError('The three original validation datasets and fixed pair files are required')
    result = {}
    for domain, path in files.items():
        original = original_validation_records if original_validation_records is not None else records
        if condition.startswith('E1-C') and original_validation_records is None:
            raise ValueError('Supply the complete frozen validation inventory before taking its pure-V2 availability view')
        pairs, _ = pairs_validate_pairs(data_load_table(path).to_dict('records'), original, domain=domain, split='validation')
        allowed = {r['image_id'] for r in records}
        if condition.startswith('E1-C'):
            pairs = [p for p in pairs if p['image_id_a'] in allowed and p['image_id_b'] in allowed]
        elif any((p['image_id_a'] not in allowed or p['image_id_b'] not in allowed for p in pairs)):
            raise ValueError('Cartesian validation pair population differs from retained validation membership')
        if not any((p['is_genuine'] for p in pairs)) or not any((p['pair_type'] == 'impostor' for p in pairs)):
            raise ValueError(f'Insufficient frozen validation pairs for {domain}')
        result[domain] = pairs
    return result

### RNG defaults

In [ ]:
def runtime_seed_everything(seed: int=42, deterministic: bool=True) -> None:
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)

## Frozen common populations and inference

In [ ]:
DATASETS=('CUST-Iris','CASIA-Interval','IITD','AMF','UTIRIS_NIR')
ORIGINAL_DATASETS=DATASETS[:3]
EXTERNAL_DATASETS=DATASETS[3:]
CONDITIONS=('E1-A','E1-B1','E1-B2','E1-C1','E1-C2')
ARCHITECTURES=('ResNet-18','SwinV2-Tiny')
METRICS=('eer','roc_auc','tar_at_far_1pct','tar_at_far_0_1pct')
KINDS=('genuine','impostor','opposite_eye_diagnostic')
verification_metrics=metrics_verification_metrics
def required_path(path, explanation):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. {explanation} No alternate asset or population is substituted.')
    return path

def read_json(path):
    return json.loads(required_path(path,'Configure the documented asset path.').read_text(encoding='utf-8'))

def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n',encoding='utf-8')

def save_table(frame,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if not isinstance(frame,pd.DataFrame):frame=pd.DataFrame(frame)
    frame.to_csv(path.with_suffix('.csv'),index=False)
    frame.to_parquet(path.with_suffix('.parquet'),index=False)
    return frame

def config():return copy.deepcopy(ANALYSIS_CONFIG)

def settings():return copy.deepcopy(STAGE_CONFIG)

def read_image(path,unused=None):
    path=required_path(path,'Supply the unchanged uint8 image/mask cache or native source.')
    a=cv2.imdecode(np.fromfile(path,np.uint8),cv2.IMREAD_UNCHANGED)
    if a is None or a.ndim!=2 or a.dtype!=np.uint8:
        raise ValueError(f'Expected original uint8 grayscale: {path}')
    return a

def read_support(path,unused=None):
    a=read_image(path)
    assert a.shape==(320,320) and np.isin(a,[0,255]).all()
    return a>0

def _record_path(value):
    return str(data_resolve_path(value,DATA_ROOT,PATH_REMAP)) if data_present(value) else None

def evaluation_rows():
    table=pd.read_parquet(required_path(EVALUATION_MANIFEST,'Supply the evaluation-only metadata asset described in ASSETS.md.'))
    result=[]
    for r in table.to_dict('records'):
        for k in ('image_id','dataset','subject_id','iris_id','eye_side','source_id','split'):r[k]=str(r[k])
        assert r['dataset'] in DATASETS
        assert r['split']==('test' if r['dataset'] in ORIGINAL_DATASETS else 'external')
        assert r['iris_id']==f"{r['dataset']}:{r['subject_id']}:{r['eye_side']}" and r['eye_side'] in ('L','R')
        for key in ('source_path','cached_path','b1_support_path','b2_support_path','c1_path','c2_path',
                    'c1_validity_path','c2_validity_path','geometry_path','coordinates_path','eyelash_footprint_path'):
            r[key]=_record_path(r.get(key))
        masks=r['mask_paths']
        if isinstance(masks,str):masks=json.loads(masks)
        r['mask_paths']=json.dumps({k:_record_path(v) for k,v in masks.items()})
        flags=r.get('quality_flags',[])
        if not isinstance(flags,str):flags=json.dumps(flags.tolist() if hasattr(flags,'tolist') else list(flags))
        r['quality_flags']=flags
        if not isinstance(r['raw_geometry_json'],str):r['raw_geometry_json']=json.dumps(r['raw_geometry_json'])
        # Original tensor-loader schema: coordinates and support stay in the same frame.
        r.update(native_path=r['source_path'],raw_path=r['cached_path'],
                 b1_status=r.get('segmentation_status','ok'),b2_status=r.get('segmentation_status','ok'),
                 geometry_version='v2',v1_fallback_used=False,v3_used=False,
                 failure_reason=r.get('polar_failure_reason',''))
        result.append(r)
    assert len({r['image_id'] for r in result})==len(result)
    assert len({r['source_id'] for r in result})==len(result),'Supply the existing exact-duplicate exclusions'
    # Use frozen membership to enforce the train/test boundary without loading train images.
    frozen=read_json(SUBJECT_SPLITS)['datasets']
    for r in result:
        if r['dataset'] in ORIGINAL_DATASETS:
            assert r['subject_id'] in frozen[r['dataset']]['test']
            assert all(r['subject_id'] not in frozen[r['dataset']][s] for s in ('train','validation'))
    return result

def manifest(dataset):
    rows=[r for r in evaluation_rows() if r['dataset']==dataset]
    if dataset in ORIGINAL_DATASETS:return sorted(rows,key=lambda r:r['image_id'])
    if any('inference_order' not in r for r in rows):
        raise ValueError('External metadata must retain the original inference_order; filename sorting is not a replacement.')
    return sorted(rows,key=lambda r:int(r['inference_order']))

def domain_views(dataset):
    rows=manifest(dataset)
    split='test' if dataset in ORIGINAL_DATASETS else 'external'
    return {c:data_condition_view(rows,c,split)[0] for c in CONDITIONS}

def load_population(dataset,population='matched'):
    assert population=='matched'
    root=MATCHED_POPULATION/dataset
    members=pd.read_parquet(required_path(root/'membership.parquet','Supply the published frozen population metadata.'))
    pairs={k:pd.read_parquet(required_path(root/f'{k}.parquet','Supply the published frozen verification pairs.')) for k in KINDS}
    available={r['image_id']:r for r in manifest(dataset)}
    records=[available[x] for x in members.image_id]
    for kind,p in pairs.items():
        assert p.pair_id.is_unique
        pairs_validate_pairs(p.to_dict('records'),records,domain=dataset,split='test' if dataset in ORIGINAL_DATASETS else 'external')
    if population=='matched':
        views=domain_views(dataset)
        actual=set.intersection(*(set(r['image_id'] for r in views[c]) for c in CONDITIONS))
        assert actual==set(members.image_id),'Do not redefine frozen common eligibility'
    return members,pairs

def load_common(dataset):return load_population(dataset,'matched')

def infer(model,dataset,job):
    assert not model.training and all(not p.requires_grad for p in model.parameters())
    loader=DataLoader(dataset,batch_size=24,shuffle=False,num_workers=0)
    ids=[];vectors=[];torch.cuda.reset_peak_memory_stats();started=time.perf_counter()
    with torch.inference_mode():
        for batch in loader:
            assert set(batch['split'])<= {'test','external'}
            with torch.autocast('cuda',dtype=torch.float16,enabled=job['config']['runtime']['amp']):
                z=model(batch['image'].cuda())
            assert z.shape==(len(batch['image_id']),256) and torch.isfinite(z).all()
            assert torch.allclose(z.norm(dim=1),torch.ones(len(z),device='cuda'),atol=1e-5)
            ids.extend(batch['image_id']);vectors.append(z.float().cpu().numpy())
    assert len(ids)==len(set(ids))
    return ids,np.concatenate(vectors),dict(seconds=time.perf_counter()-started,
        cuda_peak_allocated_bytes=torch.cuda.max_memory_allocated(),batch_size=24,
        amp=job['config']['runtime']['amp'],normalization=config()['normalization'],training=False)

def release(model):
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def score_embeddings(ids,matrix,pairs):
    return {k:pair_scores(ids,matrix,p.to_dict('records')) for k,p in pairs.items()}

def metric_artifacts(scores,target,details):
    target=Path(target);target.mkdir(parents=True,exist_ok=False)
    result=verification_metrics(scores['genuine'],scores['impostor'])
    fpr,tpr,_=roc_curve(np.r_[np.ones(len(scores['genuine'])),np.zeros(len(scores['impostor']))],
        np.r_[scores['genuine'],scores['impostor']],drop_intermediate=False)
    np.savez_compressed(target/'scores.npz',**scores)
    np.savez_compressed(target/'roc.npz',fpr=fpr,tpr=tpr)
    record=dict(details,metrics=result,score_distributions={k:distribution(v) for k,v in scores.items()})
    write_json(target/'metrics.json',record)
    return record

def start_output():
    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError('Choose a fresh OUTPUT_ROOT; recorded results and previous reproductions are never overwritten.')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    torch.set_num_threads(2);cv2.setNumThreads(1)

## Original numerical scoring and summaries

In [ ]:
def summary(frame, groups, metrics=METRICS):
    rows = []
    for keys, group in frame.groupby(groups, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        r = dict(zip(groups, keys))
        assert set(group.seed) == {42, 123, 2025} and len(group) == 3
        for metric in metrics:
            r[metric + '_mean'] = float(group[metric].mean())
            r[metric + '_sample_sd'] = float(group[metric].std(ddof=1))
            for seed in (42, 123, 2025):
                r[f'{metric}_seed{seed}'] = float(group.loc[group.seed == seed, metric].iloc[0])
        rows.append(r)
    return pd.DataFrame(rows)

def distribution(values):
    return dict(n=len(values), mean=float(np.mean(values)), sample_sd=float(np.std(values, ddof=1)), **dict(zip(('p01', 'p10', 'p50', 'p90', 'p99', 'p999', 'max'), map(float, np.quantile(values, [0.01, 0.1, 0.5, 0.9, 0.99, 0.999, 1])))))

def pair_scores(image_ids, embeddings, pairs, chunk_size=65536):
    """Same float32 cosine calculation as the original, with bounded scratch."""
    if chunk_size < 1 or len(set(image_ids)) != len(image_ids):
        raise ValueError('Positive chunk size and unique embedding IDs required')
    matrix = np.asarray(embeddings, dtype=np.float32).copy()
    if matrix.ndim != 2 or len(matrix) != len(image_ids) or (not np.isfinite(matrix).all()):
        raise ValueError('Invalid embedding matrix')
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if (norms <= 0).any():
        raise ValueError('Zero embedding')
    matrix /= norms
    index = {key: i for i, key in enumerate(image_ids)}
    result = np.empty(len(pairs), dtype=np.float32)
    for start in range(0, len(pairs), chunk_size):
        batch = pairs[start:start + chunk_size]
        a = [index[p['image_id_a']] for p in batch]
        b = [index[p['image_id_b']] for p in batch]
        result[start:start + len(batch)] = np.clip(np.einsum('ij,ij->i', matrix[a], matrix[b], dtype=np.float32), -1.0, 1.0)
    return result

## Frozen membership and pair assertions

In [ ]:
def aligned_inputs(dataset):
    members, pairs = load_common(dataset)
    rows_by_id = {r['image_id']: r for r in manifest(dataset)}
    rows = [rows_by_id[i] for i in members.image_id]
    assert len(rows) == len(set(members.image_id))
    for row in rows:
        assert row['dataset'] == dataset
        assert row['split'] == ('test' if dataset in ORIGINAL_DATASETS else 'external')
        assert row['iris_id'] == f"{dataset}:{row['subject_id']}:{row['eye_side']}"
    assert members.dataset.eq(dataset).all()
    return members, pairs, rows

def validate_pairs(members, pairs):
    by = members.set_index('image_id')
    seen = set()
    for kind, p in pairs.items():
        assert p.pair_id.is_unique and set(p.pair_id).isdisjoint(seen)
        seen.update(p.pair_id)
        a = by.loc[p.image_id_a].reset_index(drop=True)
        b = by.loc[p.image_id_b].reset_index(drop=True)
        assert (p.image_id_a != p.image_id_b).all()
        assert (a.source_id != b.source_id).all()
        assert (a.dataset == b.dataset).all()
        if kind == 'genuine':
            assert (a.iris_id == b.iris_id).all()
        elif kind == 'impostor':
            assert (a.subject_id != b.subject_id).all()
        else:
            assert kind == 'opposite_eye_diagnostic'
            assert (a.subject_id == b.subject_id).all() and (a.eye_side != b.eye_side).all()
    # Confirm genuine and primary impostors are exhaustive, not sampled anew.
    n = len(members)
    choose2 = lambda sizes: int(sum(s * (s - 1) // 2 for s in sizes))
    assert len(pairs['genuine']) == choose2(members.groupby('iris_id').size())
    assert len(pairs['impostor']) == n * (n - 1) // 2 - choose2(members.groupby('subject_id').size())

## E4 checkpoint binding and evaluation

In [ ]:
NORMALIZATION={'mean': 0.48287347719744544, 'std': 0.2741831329802273}

def native_footprint(support, geometry):
    size = geometry['content_width'], geometry['content_height']
    fraction = support.astype('float32')
    if fraction.shape != (size[1], size[0]):
        fraction = cv2.resize(fraction, size, interpolation=cv2.INTER_AREA)
    return cv2.copyMakeBorder(fraction, geometry['pad_top'], geometry['pad_bottom'],
        geometry['pad_left'], geometry['pad_right'], cv2.BORDER_CONSTANT, value=0)

def content_mask(geometry):
    result = np.zeros((320, 320), bool)
    y, x = geometry['pad_top'], geometry['pad_left']
    result[y:y+geometry['content_height'], x:x+geometry['content_width']] = True
    return result

def removal_mask(native_annulus, geometry, original_annulus):
    if native_annulus.dtype != np.bool_:
        raise ValueError('Expected native Boolean annulus')
    fraction = native_footprint(native_annulus, geometry)
    removal = original_annulus | (fraction > 0)
    content = content_mask(geometry)
    if removal.shape != (320, 320) or np.any(removal & ~content):
        raise ValueError('Removal crosses original image geometry')
    if np.any((fraction > 0) & ~removal):
        raise ValueError('Native annulus sampling contribution remains')
    return removal, fraction

def mean_fill(normalized, mask, content):
    assert normalized.dtype == torch.float32 and normalized.shape == (1, 320, 320)
    assert mask.dtype == np.bool_ and not (mask & ~content).any()
    output = normalized.clone()
    selection = torch.from_numpy(mask)
    output[:, selection] = 0.
    assert torch.equal(output[:, ~selection], normalized[:, ~selection])
    return output

def tensor_from_pixels(pixels, mask, content, augmentation=None):
    if pixels.dtype != np.uint8 or pixels.shape != (320, 320):
        raise ValueError('Use original standardized grayscale cache; no resize')
    image = torch.from_numpy(np.ascontiguousarray(pixels[None])).float().div(255)
    if augmentation is not None:
        # Discard texture before contrast computes the image mean.
        image[:, torch.from_numpy(mask)] = NORMALIZATION['mean']
        image = augmentation(image)
    normalized = (image-NORMALIZATION['mean'])/NORMALIZATION['std']
    return mean_fill(normalized, mask, content)

def attach_removal_masks(records, manifest, mask_root, path_remap=None):
    """Join the frozen train/validation mask asset without changing membership."""
    table = pd.read_parquet(manifest)
    if not table.image_id.is_unique:
        raise ValueError('Duplicate image in complement mask manifest')
    active = [r for r in records if r['split'] in ('train', 'validation')]
    if set(table.image_id) != {r['image_id'] for r in active}:
        raise ValueError('Complement manifest must match frozen train/validation exactly')
    by_id = table.set_index('image_id').to_dict('index')
    result = []
    for record in records:
        row = dict(record)
        if row['split'] in ('train', 'validation'):
            selected = by_id[row['image_id']]
            if 'split' in selected and selected['split'] != row['split']:
                raise ValueError('Complement mask split mismatch')
            value = str(selected['removal_mask_path']).replace('\\', '/')
            for old, new in sorted((path_remap or {}).items(), key=lambda p:-len(str(p[0]))):
                prefix = str(old).replace('\\', '/').rstrip('/')
                if value == prefix or value.startswith(prefix+'/'):
                    value = str(Path(new)/value[len(prefix):].lstrip('/'))
                    break
            path = Path(value)
            row['e4_removal_mask_path'] = (path if path.is_absolute() else Path(mask_root)/path).resolve()
        result.append(row)
    return result

class ComplementEvaluationDataset(Dataset):
    def __init__(self,rows):
        self.records=list(rows)
        if any(r['split']!=('test' if r['dataset'] in ORIGINAL_DATASETS else 'external') for r in rows):
            raise ValueError('Evaluation cannot load train/validation images')

    def __len__(self):return len(self.records)

    def __getitem__(self,index):
        row=self.records[index]
        geometry=json.loads(row['raw_geometry_json'])
        paths=json.loads(row['mask_paths'])
        native={k:read_image(paths[k])>0 for k in ('iris','pupil','annulus')}
        expected=(int(row['height']),int(row['width']))
        assert all(m.shape==expected for m in native.values())
        assert np.array_equal(native['annulus'],native['iris']&~native['pupil'])
        old=read_image(row['b1_support_path'])>0
        assert np.array_equal(old,preprocessing_transform_support(native['annulus'],geometry))
        removal,fraction=removal_mask(native['annulus'],geometry,old)
        image=tensor_from_pixels(read_image(row['cached_path']),removal,content_mask(geometry))
        assert not ((fraction>0)&~removal).any() and torch.all(image[:,removal]==0)
        return dict(image=image,**{k:row[k] for k in ('image_id','dataset','subject_id','eye_side','iris_id','split')})


def e4_jobs():
    rows=read_json(E4_CHECKPOINT_INDEX)['jobs']
    expected={(a,c,s) for a in ARCHITECTURES for c in ('A-complement-mean-strict',) for s in (42,123,2025)}
    assert len(rows)==6 and {(r['architecture'],r['condition'],r['seed']) for r in rows}==expected
    return rows

def load_e4_model(job):
    assert torch.cuda.is_available(), 'Use the recorded CUDA/AMP policy; no implicit CPU fallback'
    cfg=config_frozen_config(job['architecture'],job['reference_condition'],job['seed'],'cuda')
    assert cfg['augmentation']['enabled'] is True
    runtime_seed_everything(job['seed'],cfg['runtime']['deterministic_algorithms'])
    path=required_path(CHECKPOINT_ROOT/job['checkpoint'],'Supply this exact E4 validation-selected best checkpoint')
    with path.open('rb') as handle:
        assert hashlib.file_digest(handle,'sha256').hexdigest()==job['checkpoint_sha256']
    state=torch.load(path,map_location='cpu',weights_only=False)
    assert state['epoch']==state['best_epoch']==job['selected_epoch']
    assert state['config']['seed']==job['seed']
    assert state['config']['architecture']['name']==job['config']['architecture']['name']
    assert state['config']['data']['representation']==job['condition']
    assert state['config']['augmentation']['enabled'] is True
    model=models_build_model(cfg,pretrained=False)
    model.load_state_dict(state['model_state'],strict=True)
    return model.eval().requires_grad_(False).cuda()

def evaluate_e4():
    start_output()
    populations={}
    for dataset in DATASETS:
        members,pairs,rows=aligned_inputs(dataset)
        validate_pairs(members,pairs)
        populations[dataset]=(members,pairs,rows)
    records=[]
    for job in e4_jobs():
        model=None
        for dataset in DATASETS:
            members,pairs,rows=populations[dataset]
            if INFERENCE_SOURCE=='saved_embeddings':
                path=required_path(E4_EMBEDDING_ROOT/dataset/job['name']/'embeddings.npz',
                    'Supply the exact completed E4 cache or explicitly enable recompute')
                with np.load(path,allow_pickle=False) as z:
                    ids=z['image_ids'].astype(str).tolist(); matrix=z['embeddings'].copy()
                runtime={'saved_embeddings':True,'training':False}
            elif INFERENCE_SOURCE=='recompute':
                if model is None:model=load_e4_model(job)
                data=ComplementEvaluationDataset(rows)
                ids,matrix,runtime=infer(model,data,job)
            else:
                raise ValueError('Choose saved_embeddings or recompute explicitly')
            assert ids==members.image_id.tolist(),'Preserve original E4 matched-membership inference order'
            assert matrix.dtype==np.float32 and matrix.shape==(len(ids),256) and np.isfinite(matrix).all()
            np.testing.assert_allclose(np.linalg.norm(matrix,axis=1),1,atol=1e-5,rtol=0)
            scores=score_embeddings(ids,matrix,pairs)
            target=OUTPUT_ROOT/dataset/job['name']
            record=metric_artifacts(scores,target,dict(dataset=dataset,architecture=job['architecture'],
                condition=job['condition'],seed=job['seed'],selected_epoch=job['selected_epoch'],
                checkpoint_sha256=job['checkpoint_sha256'],inference=runtime,training=False,
                strict_native_annulus_footprint=True,threshold_fitting=False))
            records.append(dict(dataset=dataset,architecture=job['architecture'],condition=job['condition'],
                seed=job['seed'],selected_epoch=job['selected_epoch'],checkpoint_sha256=job['checkpoint_sha256'],
                **{m:record['metrics'][m] for m in (*METRICS,'num_genuine','num_impostor','far_resolution')}))
            print(dataset,job['name'],record['metrics']['eer'])
        if model is not None:release(model)
    frame=save_table(records,OUTPUT_ROOT/'individual_metrics')
    save_table(summary(frame,['dataset','architecture','condition']),OUTPUT_ROOT/'three_seed_summary')
    return frame


## Recorded results and optional replay

In [ ]:
if SHOW_RECORDED_RESULTS:
    display(pd.read_csv(RECORDED_RESULTS/'matched_population_counts.csv'))
    display(pd.read_csv(RECORDED_RESULTS/'three_seed_summary.csv'))
    display(pd.read_csv(RECORDED_RESULTS/'paired_changes_vs_raw_A_summary.csv'))
if RUN_ANALYSIS:
    reproduced=evaluate_e4()
else:
    print('No inference or score recomputation requested; original results remain unchanged.')

## Comparison and table reproduction

Compare independently trained complement checkpoints with original raw-A checkpoints, paired by seed on exactly matched images/pairs. See [the completed report](REPORT.md), [paired summaries](Results/paired_changes_vs_raw_A_summary.csv) and the portable [paper-table builder](../Manuscript%20Tables/build_tables.py). This input retains pupil and other ocular samples, subject to predicted masks; it is not a pure-periocular or causal-shortcut claim.